In [ ]:
import torch
import matplotlib.pyplot as plt
import sys
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), "../../"))
sys.path.append(project_root)

import models
from models import baselines
from utils.training import train_variational_model
from utils.data_utils import obtain_me_a_nice_sawtooth_dataset_please, obtain_me_a_nice_heaviside_dataset_please, obtain_me_a_nice_gp_dataset_please, ctxt_trgt_split
from utils.mcmc_utils import autocorrelation_array

device = torch.device('cpu')
torch.set_default_device(device)
torch.set_default_dtype(torch.float64)

%load_ext autoreload
%autoreload 2

In [ ]:
X, y = obtain_me_a_nice_gp_dataset_please(l=0.5, kernel='se', x_range=[-2.0, 2.0], n_range=[10,11])
lik = models.GaussianLikelihood(1, sigma_y=0.05)
alg = 'lmc'
model = baselines.HMC_BNN(x_dim=1,
                          y_dim=1,
                          likelihood=lik,
                          hidden_dims=[48,48],
                          scale_prior=True,
                          nonlinearity=torch.nn.SiLU())


xs = torch.linspace(-4.0, 4.0, 200).unsqueeze(-1)
if alg == 'hmc':
    step_size = 1e-3
    steps = 5_000
    burn = 2_000
    thin = 50
    leapfrog_steps = 100
else:
    step_size = 1e-3
    steps = 250_000
    burn = 50_000
    thin = 5_000
    leapfrog_steps = 1
# init_samples, _ = baselines.run_mcmc(model, X, y, algorithm='hmc', steps=50, step_size=5e-4, metropolis_adjusted=True, leapfrog_steps=100)
# init_samp = init_samples[-1,:]
init_samp, _ = model.get_map_sln(X, y)
raw_samples, training_metrics = baselines.run_mcmc(model,
                                                    X,
                                                    y,
                                                    algorithm='hmc', # easier to tune step sizes etc if we run LMC as HMC with a single leapfrog step.
                                                    steps=steps,
                                                    step_size=step_size,
                                                    minibatch_size=None, # full-batch
                                                    metropolis_adjusted=True,
                                                    leapfrog_steps=leapfrog_steps,
                                                    init_W=init_samp,
                                                    )
            
burned_in_samples = raw_samples[burn:] # do burn-in and thinning here
samples = burned_in_samples[::thin]

# training_metrics['autocorrelation'] = autocorrelation_array(raw_samples, max_lag=50)            

In [ ]:
with torch.no_grad():
    pred_samps = model.batch_forward(xs, samples)
    num_samples = pred_samps.shape[0]

fig, axes = plt.subplots(1, len(training_metrics), figsize=(3*len(training_metrics), 1))
omitted_steps = 0
for i, (key, value) in enumerate(training_metrics.items()):
    axes[i].plot(value[omitted_steps:])
    axes[i].set_xlabel(key)
    axes[i].grid()
plt.show()

###### evaluation code common to all models.
plt.plot(xs.unsqueeze(0).repeat((pred_samps.shape[0], 1, 1)).squeeze(-1).T.cpu(), pred_samps.squeeze(-1).T.cpu(), linewidth=0.5, color='C0', alpha=0.5)
plt.scatter(X, y, color='C1', zorder=10000)
plt.grid()
lim = [-4.0, 4.0]
plt.xlim(lim)
plt.ylim(lim)
plt.show()